# Notebook 01 — Experimentos de Machine Learning (FordA)

Classificação binária com modelos ML clássicos sobre features wavelet extraídas do dataset FordA.

**Modelos:** LinearSVC, SGDClassifier, LogisticRegression, RandomForest, XGBoost, LightGBM, CatBoost, Stacking Ensemble

**Pipeline:** Features Wavelet → VarianceThreshold → StandardScaler → RandomizedSearchCV (StratifiedKFold) → Avaliação (Accuracy, F1, Precision, Recall, AUC-ROC)

In [1]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import joblib
from pathlib import Path

from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold, mutual_info_classif
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import accuracy_score
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

# Imports locais
sys.path.insert(0, str(Path.cwd()))
from config.experiment_config import (
    DATA_DIR, RESULTS_DIR, MODELS_DIR, SEED,
    WAVELET_CONFIG, ML_MODELS_CONFIG, ML_SEARCH_CONFIG,
    ML_FEATURE_CONFIG, VALIDATION_CONFIG,
    build_param_dist,
)
from src.feature_extraction import WaveletFeatureExtractor
from src.evaluation import ClassificationEvaluator, ResultsManager
from src.visualization import ExperimentVisualizer
from src.models import (
    create_linear_svc, create_sgd_classifier, create_logistic_regression,
    create_rf_classifier, create_xgb_classifier, create_lgbm_classifier,
    create_catboost_classifier,
)

ML_RESULTS_DIR = RESULTS_DIR / "ml_experiments"
ML_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"DATA_DIR: {DATA_DIR}")
print(f"RESULTS_DIR: {ML_RESULTS_DIR}")

I0000 00:00:1773033250.638588 1784026 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


DATA_DIR: /home/felipeteodoro/projetos/LearnableWaveletLayer/tests/ford-a/data
RESULTS_DIR: /home/felipeteodoro/projetos/LearnableWaveletLayer/tests/ford-a/results/ml_experiments


## 1. Carregar Dados

In [2]:
X_train = np.load(DATA_DIR / "X_train.npy")
y_train = np.load(DATA_DIR / "y_train.npy")
X_val = np.load(DATA_DIR / "X_val.npy")
y_val = np.load(DATA_DIR / "y_val.npy")
X_test = np.load(DATA_DIR / "X_test.npy")
y_test = np.load(DATA_DIR / "y_test.npy")

print(f"Treino:    X={X_train.shape}, y={y_train.shape} (classe 1: {y_train.mean():.2%})")
print(f"Validação: X={X_val.shape},   y={y_val.shape} (classe 1: {y_val.mean():.2%})")
print(f"Teste:     X={X_test.shape},  y={y_test.shape} (classe 1: {y_test.mean():.2%})")

Treino:    X=(3060, 500), y=(3060,) (classe 1: 48.73%)
Validação: X=(541, 500),   y=(541,) (classe 1: 48.80%)
Teste:     X=(1320, 500),  y=(1320,) (classe 1: 48.41%)


## 2. Extração de Features (Wavelet Stats + Coeficientes Brutos)

In [ ]:
wfe = WaveletFeatureExtractor(
    wavelet=WAVELET_CONFIG["wavelet_type"],
    level=WAVELET_CONFIG["decomposition_level"],
    mode=WAVELET_CONFIG["mode"],
)

# Carregar features pré-extraídas (stats + raw coefficients, do notebook 00) ou re-extrair
feat_train_path = DATA_DIR / "X_train_wavelet_features.npy"
if feat_train_path.exists():
    X_train_feat = np.load(feat_train_path)
    X_val_feat = np.load(DATA_DIR / "X_val_wavelet_features.npy")
    X_test_feat = np.load(DATA_DIR / "X_test_wavelet_features.npy")
    print(f"✓ Features carregadas do cache")
else:
    # Extrair stats + raw coefficients
    X_train_stats = wfe.extract_features(X_train)
    X_val_stats = wfe.extract_features(X_val)
    X_test_stats = wfe.extract_features(X_test)
    X_train_coeffs = wfe.get_coefficients(X_train)
    X_val_coeffs = wfe.get_coefficients(X_val)
    X_test_coeffs = wfe.get_coefficients(X_test)
    X_train_feat = np.hstack([X_train_stats, X_train_coeffs])
    X_val_feat = np.hstack([X_val_stats, X_val_coeffs])
    X_test_feat = np.hstack([X_test_stats, X_test_coeffs])
    print(f"✓ Features extraídas: stats + raw wavelet coefficients")

# Nomes das features (stats + coeff indices)
stat_names = wfe.get_feature_names()
n_coeff = X_train_feat.shape[1] - len(stat_names)
coeff_names = [f"wcoeff_{i}" for i in range(n_coeff)]
feature_names = stat_names + coeff_names
print(f"Shape features: treino={X_train_feat.shape}, val={X_val_feat.shape}, teste={X_test_feat.shape}")
print(f"Total: {len(feature_names)} features ({len(stat_names)} stats + {n_coeff} coeffs)")

✓ Features carregadas do cache
Shape features: treino=(3060, 51), val=(541, 51), teste=(1320, 51)
Total de features: 51


In [4]:
# Tratar NaN/Inf
for arr in [X_train_feat, X_val_feat, X_test_feat]:
    np.nan_to_num(arr, copy=False, nan=0.0, posinf=0.0, neginf=0.0)

# VarianceThreshold
var_sel = VarianceThreshold(threshold=ML_FEATURE_CONFIG["variance_threshold"])
X_train_sel = var_sel.fit_transform(X_train_feat)
X_val_sel = var_sel.transform(X_val_feat)
X_test_sel = var_sel.transform(X_test_feat)
selected_mask = var_sel.get_support()
selected_names = [f for f, m in zip(feature_names, selected_mask) if m]
print(f"Features após VarianceThreshold: {X_train_sel.shape[1]} (de {X_train_feat.shape[1]})")

# StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_sel)
X_val_scaled = scaler.transform(X_val_sel)
X_test_scaled = scaler.transform(X_test_sel)

# Combinar treino + val para CV
X_search = np.vstack([X_train_scaled, X_val_scaled])
y_search = np.concatenate([y_train, y_val])
print(f"Dados para busca (treino+val): {X_search.shape}")

# Mutual Information
mi_scores = mutual_info_classif(
    X_train_scaled, y_train,
    n_neighbors=ML_FEATURE_CONFIG["mi_n_neighbors"],
    random_state=SEED
)
mi_df = pd.DataFrame({"feature": selected_names, "mi_score": mi_scores})
mi_df = mi_df.sort_values("mi_score", ascending=False)
top_k = ML_FEATURE_CONFIG["mi_top_k"]
print(f"\nTop {top_k} features por Mutual Information:")
print(mi_df.head(top_k).to_string(index=False))

Features após VarianceThreshold: 51 (de 51)
Dados para busca (treino+val): (3601, 51)

Top 15 features por Mutual Information:
                feature  mi_score
detail_1_zero_crossings  0.239484
detail_2_mean_crossings  0.238945
detail_2_zero_crossings  0.233877
detail_1_mean_crossings  0.229753
        detail_1_energy  0.214013
           detail_1_rms  0.213944
           detail_1_std  0.213476
           detail_1_var  0.213401
           detail_2_var  0.207455
           detail_2_std  0.207378
           detail_2_rms  0.205971
        detail_2_energy  0.205929
           detail_2_mad  0.187273
           detail_2_iqr  0.183881
  detail_1_peak_to_peak  0.163389


## 3. Configuração de Experimentos

In [5]:
evaluator = ClassificationEvaluator()
results_mgr = ResultsManager(ML_RESULTS_DIR)
viz = ExperimentVisualizer()

all_results = {}
all_cv_results = {}
fitted_models = {}

cv = StratifiedKFold(
    n_splits=VALIDATION_CONFIG["n_folds"],
    shuffle=VALIDATION_CONFIG["shuffle"],
    random_state=SEED,
)


def run_experiment(model_name, model_factory, config_key):
    """Executa RandomizedSearchCV + avaliação no teste para um modelo."""
    print(f"\n{'=' * 60}")
    print(f"  {model_name}")
    print(f"{'=' * 60}")

    cfg = ML_MODELS_CONFIG[config_key]
    param_dist = build_param_dist(cfg["param_dist"])
    model = model_factory(cfg.get("model_kwargs", {}))

    search = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_dist,
        n_iter=cfg["n_iter"],
        cv=cv,
        scoring=ML_SEARCH_CONFIG["scoring"],
        n_jobs=ML_SEARCH_CONFIG["n_jobs"],
        verbose=ML_SEARCH_CONFIG["verbose"],
        random_state=SEED,
        refit=True,
    )
    search.fit(X_search, y_search)

    best = search.best_estimator_
    y_pred = best.predict(X_test_scaled)

    # Probabilidades (se disponível)
    y_prob = None
    if hasattr(best, "predict_proba"):
        y_prob = best.predict_proba(X_test_scaled)[:, 1]
    elif hasattr(best, "decision_function"):
        y_prob = best.decision_function(X_test_scaled)

    metrics = evaluator.evaluate(y_test, y_pred, y_prob)

    print(f"\nBest params: {search.best_params_}")
    print(f"CV Accuracy: {search.best_score_:.4f}")
    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}")

    # Salvar
    all_results[model_name] = metrics
    all_cv_results[model_name] = pd.DataFrame(search.cv_results_)
    fitted_models[model_name] = best

    results_mgr.log_experiment(
        experiment_name="ml_experiments",
        model_name=model_name,
        metrics=metrics,
        config={"best_params": str(search.best_params_), "cv_score": search.best_score_},
    )
    results_mgr.save_predictions(f"ml_{model_name}", y_test, y_pred, y_prob)
    joblib.dump(best, MODELS_DIR / f"ml_{model_name}.pkl")

    return best, metrics

print("✓ Configuração pronta")
print(f"  CV: {VALIDATION_CONFIG['n_folds']}-fold StratifiedKFold")
print(f"  Scoring: {ML_SEARCH_CONFIG['scoring']}")

✓ Configuração pronta
  CV: 5-fold StratifiedKFold
  Scoring: accuracy


## 4. Experimento 1: LinearSVC

In [6]:
best_linearsvc, metrics_linearsvc = run_experiment(
    "LinearSVC", create_linear_svc, "LinearSVC"
)


  LinearSVC
Fitting 5 folds for each of 15 candidates, totalling 75 fits



Best params: {'estimator__C': np.float64(11.435780278433395), 'estimator__loss': 'hinge'}
CV Accuracy: 0.8248
  accuracy: 0.8205
  f1: 0.8200
  precision: 0.7965
  recall: 0.8451
  auc_roc: 0.9148
  log_loss: 0.3719
  specificity: 0.7974


## 5. Experimento 2: SGDClassifier

In [7]:
best_sgdclassifier, metrics_sgdclassifier = run_experiment(
    "SGDClassifier", create_sgd_classifier, "SGDClassifier"
)


  SGDClassifier
Fitting 5 folds for each of 20 candidates, totalling 100 fits

Best params: {'alpha': np.float64(0.004073745196058387), 'l1_ratio': np.float64(0.9385527090157502), 'learning_rate': 'optimal', 'loss': 'hinge', 'penalty': 'l1'}
CV Accuracy: 0.8206
  accuracy: 0.8235
  f1: 0.8268
  precision: 0.7875
  recall: 0.8701
  auc_roc: 0.9165
  log_loss: nan
  specificity: 0.7797


## 6. Experimento 3: LogisticRegression

In [8]:
best_logisticregression, metrics_logisticregression = run_experiment(
    "LogisticRegression", create_logistic_regression, "LogisticRegression"
)


  LogisticRegression
Fitting 5 folds for each of 15 candidates, totalling 75 fits

Best params: {'C': np.float64(0.1767016940294795), 'l1_ratio': np.float64(0.9507143064099162), 'penalty': 'elasticnet'}
CV Accuracy: 0.8239
  accuracy: 0.8250
  f1: 0.8262
  precision: 0.7957
  recall: 0.8592
  auc_roc: 0.9155
  log_loss: 0.3775
  specificity: 0.7930


## 7. Experimento 4: RandomForest

In [ ]:
best_randomforest, metrics_randomforest = run_experiment(
    "RandomForest", create_rf_classifier, "RandomForest"
)

## 8. Experimento 5: XGBoost

In [ ]:
best_xgboost, metrics_xgboost = run_experiment(
    "XGBoost", create_xgb_classifier, "XGBoost"
)

## 9. Experimento 6: LightGBM

In [ ]:
best_lightgbm, metrics_lightgbm = run_experiment(
    "LightGBM", create_lgbm_classifier, "LightGBM"
)

## 10. Experimento 7: CatBoost

In [ ]:
best_catboost, metrics_catboost = run_experiment(
    "CatBoost", create_catboost_classifier, "CatBoost"
)

## 11. Experimento 8: Stacking Ensemble

In [ ]:
print(f"\n{'=' * 60}")
print("  Stacking Ensemble")
print(f"{'=' * 60}")

# Usar os melhores modelos já treinados como base
base_learners = [
    ("rf", fitted_models.get("RandomForest", create_rf_classifier())),
    ("xgb", fitted_models.get("XGBoost", create_xgb_classifier())),
    ("lgbm", fitted_models.get("LightGBM", create_lgbm_classifier())),
    ("catboost", fitted_models.get("CatBoost", create_catboost_classifier())),
]

stacking = StackingClassifier(
    estimators=base_learners,
    final_estimator=LogisticRegressionCV(cv=3, max_iter=5000, random_state=SEED),
    n_jobs=ML_MODELS_CONFIG["Stacking"]["base_n_jobs"],
    passthrough=ML_MODELS_CONFIG["Stacking"]["model_kwargs"]["passthrough"],
)

stacking.fit(X_search, y_search)

y_pred_stack = stacking.predict(X_test_scaled)
y_prob_stack = stacking.predict_proba(X_test_scaled)[:, 1]
metrics_stack = evaluator.evaluate(y_test, y_pred_stack, y_prob_stack)

print(f"\nStacking Ensemble:")
for k, v in metrics_stack.items():
    print(f"  {k}: {v:.4f}")

all_results["Stacking"] = metrics_stack
fitted_models["Stacking"] = stacking
results_mgr.log_experiment(
    experiment_name="ml_experiments",
    model_name="Stacking",
    metrics=metrics_stack,
    config={"base_learners": [n for n, _ in base_learners]},
)
results_mgr.save_predictions("ml_Stacking", y_test, y_pred_stack, y_prob_stack)
joblib.dump(stacking, MODELS_DIR / "ml_Stacking.pkl")

## 12. Comparação dos Resultados

In [ ]:
comparison_df = evaluator.compare_models(all_results)
comparison_df = comparison_df.sort_values("accuracy", ascending=False)
print("\n" + "=" * 80)
print("COMPARAÇÃO DOS MODELOS ML")
print("=" * 80)
print(comparison_df.to_string())

# Salvar
comparison_df.to_csv(ML_RESULTS_DIR / "ml_comparison.csv")

# Salvar CV results de todos os modelos
for model_name, cv_df in all_cv_results.items():
    cv_df.to_csv(ML_RESULTS_DIR / f"cv_results_{model_name}.csv", index=False)

print(f"\n✓ Resultados salvos em {ML_RESULTS_DIR}")

In [ ]:
# Gráfico de comparação
metrics_to_plot = ["accuracy", "f1", "precision", "recall", "auc_roc"]
fig, axes = plt.subplots(1, len(metrics_to_plot), figsize=(5 * len(metrics_to_plot), 6))

model_names = comparison_df.index.tolist()
colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(model_names)))

for ax, metric in zip(axes, metrics_to_plot):
    values = comparison_df[metric].values
    bars = ax.barh(range(len(model_names)), values, color=colors)
    ax.set_yticks(range(len(model_names)))
    ax.set_yticklabels(model_names)
    ax.set_xlabel(metric.upper())
    ax.set_title(metric.upper())
    ax.invert_yaxis()

    for bar, val in zip(bars, values):
        ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height() / 2,
                f"{val:.4f}", va="center", fontsize=8)

fig.suptitle("Comparação de Modelos ML — FordA", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(ML_RESULTS_DIR / "ml_comparison_chart.png", dpi=150, bbox_inches="tight")
plt.show()

## 13. Análise do Melhor Modelo

In [ ]:
best_model_name = comparison_df.index[0]
best_model = fitted_models[best_model_name]
print(f"Melhor modelo: {best_model_name}")
print(f"  Accuracy: {comparison_df.loc[best_model_name, 'accuracy']:.4f}")
print(f"  F1: {comparison_df.loc[best_model_name, 'f1']:.4f}")

# Confusion Matrix
y_pred_best = best_model.predict(X_test_scaled)
y_prob_best = None
if hasattr(best_model, "predict_proba"):
    y_prob_best = best_model.predict_proba(X_test_scaled)[:, 1]
elif hasattr(best_model, "decision_function"):
    y_prob_best = best_model.decision_function(X_test_scaled)

viz.plot_confusion_matrix(
    y_test, y_pred_best,
    title=f"Confusion Matrix — {best_model_name}",
    save_path=ML_RESULTS_DIR / f"confusion_matrix_{best_model_name}.png",
)
plt.show()

# ROC Curve
if y_prob_best is not None:
    viz.plot_roc_curve(
        y_test, y_prob_best,
        title=f"ROC Curve — {best_model_name}",
        save_path=ML_RESULTS_DIR / f"roc_curve_{best_model_name}.png",
    )
    plt.show()

## 14. Feature Importance (modelos baseados em árvore)

In [ ]:
tree_models = {
    name: model for name, model in fitted_models.items()
    if name in ["RandomForest", "XGBoost", "LightGBM", "CatBoost"]
}

if tree_models:
    n_models = len(tree_models)
    fig, axes = plt.subplots(1, n_models, figsize=(6 * n_models, 8))
    if n_models == 1:
        axes = [axes]

    top_k = 15
    for ax, (name, model) in zip(axes, tree_models.items()):
        if hasattr(model, "feature_importances_"):
            importances = model.feature_importances_
        else:
            ax.set_title(f"{name} — sem feature_importances_")
            continue

        idx = np.argsort(importances)[::-1][:top_k]
        ax.barh(range(top_k), importances[idx])
        ax.set_yticks(range(top_k))
        ax.set_yticklabels([selected_names[i] if i < len(selected_names) else f"f_{i}" for i in idx])
        ax.invert_yaxis()
        ax.set_title(f"Feature Importance — {name}")
        ax.set_xlabel("Importância")

    plt.tight_layout()
    plt.savefig(ML_RESULTS_DIR / "feature_importance_trees.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Nenhum modelo tree-based treinado.")

## 15. ROC Curves — Todos os Modelos

In [ ]:
# Curvas ROC de todos os modelos
roc_data = {}
for name, model in fitted_models.items():
    if hasattr(model, "predict_proba"):
        probs = model.predict_proba(X_test_scaled)[:, 1]
    elif hasattr(model, "decision_function"):
        probs = model.decision_function(X_test_scaled)
    else:
        continue
    roc_data[name] = probs

if roc_data:
    viz.plot_multi_roc(
        y_test, roc_data,
        title="ROC Curves — Todos os Modelos ML",
        save_path=ML_RESULTS_DIR / "roc_curves_all_models.png",
    )
    plt.show()

## 16. Resumo

In [ ]:
print("=" * 60)
print("RESUMO — Experimentos ML (FordA)")
print("=" * 60)
print(f"Modelos avaliados: {len(all_results)}")
print(f"Melhor modelo: {best_model_name}")
print(f"  Accuracy: {comparison_df.loc[best_model_name, 'accuracy']:.4f}")
print(f"  F1:       {comparison_df.loc[best_model_name, 'f1']:.4f}")
print(f"  AUC-ROC:  {comparison_df.loc[best_model_name, 'auc_roc']:.4f}")
print(f"Features utilizadas: {X_train_scaled.shape[1]}")
print(f"Resultados em: {ML_RESULTS_DIR}")